# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

- Gymnasium reemplaza la clase `Environment` de CheeseWorld: es la librería que define el ambiente (`gym.make("FrozenLake-v1", ...)`), maneja el estado interno, calcula las recompensas y transiciones, y expone la interfaz estándar `env.reset()` / `env.step(action)` para interactuar con él.
- El agente ya no es una clase separada como `QLearningAgent`. Aquí está representado de forma implícita, repartido en dos piezas: la **Q-table** (`Q`, la "memoria" o conocimiento del agente sobre qué tan buena es cada acción en cada estado) y las **funciones** (`epsilon_greedy_policy`, `greedy_policy`, `train_q_learning`) que definen cómo el agente decide y aprende a partir de esa tabla.
- En otras palabras, se pasó de una arquitectura orientada a objetos (una clase que encapsula el comportamiento del agente) a una más funcional/procedural, donde el "agente" es simplemente el conjunto de la Q-table más las funciones que operan sobre ella.

## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

1. **Filas:** debe tener tantas filas como estados existan en el ambiente, es decir, `env.observation_space.n`. En FrozenLake 4x4 son 16 estados (una fila por cada celda de la cuadrícula, numeradas de 0 a 15).
2. **Columnas:** debe tener tantas columnas como acciones posibles, `env.action_space.n`. En FrozenLake son 4 (Left, Down, Right, Up).
3. **Qué representa una celda Q[s,a]:** es el valor estimado (retorno esperado) de tomar la acción `a` estando en el estado `s`, y a partir de ahí seguir la mejor política posible. No es la recompensa inmediata, sino una estimación acumulada que incluye recompensas futuras descontadas por `gamma`.


In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

- Que todos los valores sean cero **no significa que todas las acciones sean malas**; significa que el agente todavía no tiene ninguna información sobre el ambiente. Cero es simplemente el valor inicial por falta de experiencia, no una evaluación negativa. Después de las primeras actualizaciones, los valores empezarán a reflejar qué tan buenas o malas son realmente las acciones.
- Si varias acciones tienen exactamente el mismo valor (por ejemplo, todas en cero al inicio), `np.argmax` por sí solo siempre devolvería la primera en orden, generando un sesgo hacia esa acción. Por eso el código usa `np.flatnonzero(Qtable[state] == max_q)` seguido de `np.random.choice(...)`: identifica **todas** las acciones empatadas en el máximo y elige una al azar entre ellas, evitando ese sesgo (esto se ve en `greedy_policy`, definida en la celda 2).


## 3. Ejecutar una transición


In [4]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 0
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(s,a,r,s') = (\text{state}, \text{action}, \text{reward}, \text{next\_state})
$$

Reemplaza cada valor con lo que imprimió tu celda anterior (por ejemplo, si el `print` mostró `state = 0`, `action = 2`, `reward = 0.0`, `next_state = 1`, la tupla sería `(0, 2, 0.0, 1)`). Como `env.reset()` no usa una semilla fija y la acción se eligió con `epsilon=1.0` (100% aleatoria), estos valores cambian cada vez que ejecutas la celda — usa los que te salieron a ti.

**¿De cuál elemento no disponíamos directamente en Value Iteration?**

En Value Iteration no interactuamos con el ambiente para obtener una experiencia real: usamos el **modelo** del ambiente (las probabilidades de transición y la función de recompensa, conocidas de antemano) para calcular el valor esperado sobre *todos* los posibles estados siguientes, ponderados por su probabilidad. Es decir, no dependíamos de una recompensa `r` ni un estado siguiente `s'` obtenidos por acción real — eran calculados a partir del modelo completo, no muestreados de una interacción real como aquí. Esa es la diferencia clave entre un método *model-based* (Value Iteration) y uno *model-free* (Q-Learning), que aprende únicamente de experiencias reales `(s,a,r,s')`.


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** _______

**Valores Q:**

- Left:
- Down:
- Right:
- Up:

¿Cuál acción seleccionaría:

$$
\arg\max_a Q(s,a)
$$

?

**Interpretación:**

>


## 5. Evaluar la política aprendida


In [9]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

>


## 6. Experimento: FrozenLake estocástico


In [10]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.194
Std reward : 0.395


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

- Ambiente:
- Algoritmo:


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

>

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

>

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

>

**4. ¿Por qué Q-Learning es model-free?**

>
